# InsightFlow Executive — Fine-tuning NLP Models

**Fine-tune XLM-RoBERTa (sentiment) + distilRoBERTa (émotion) sur le dataset InsightFlow**

| | Sentiment | Émotion |
|---|---|---|
| **Modèle base** | cardiffnlp/twitter-xlm-roberta-base-sentiment | j-hartmann/emotion-english-distilroberta-base |
| **Classes** | POSITIVE / NEUTRAL / NEGATIVE | frustration / concern / urgency / neutral / satisfaction |
| **Dataset** | insightflow_synthetic_full.csv (~13k samples) | insightflow_synthetic_full.csv (~5.9k avec emotion_label) |
| **GPU** | T4 (Colab gratuit) | T4 (Colab gratuit) |
| **Durée estimée** | ~25 min | ~15 min |

> **Important** : Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Étape 1 : Vérifier le GPU ─────────────────────────────────
import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠ Pas de GPU — va dans Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Étape 2 : Installer les dépendances ───────────────────────
!pip install -q transformers datasets torch scikit-learn accelerate evaluate

In [ ]:
# ── Étape 3 : Cloner le repo GitHub ───────────────────────────
# Option A (recommandé) — cloner directement depuis GitHub
!git clone https://github.com/sarrahafsi/InsightFlowExecutive.git
%cd InsightFlowExecutive
!ls ml/dataset/

In [ ]:
# ── Étape 3 (alternative) : Upload manuel des fichiers ────────
# Si le repo est privé ou tu veux uploader manuellement

# from google.colab import files
# import os
# os.makedirs('InsightFlowExecutive/ml/dataset', exist_ok=True)
# os.makedirs('InsightFlowExecutive/ml/models', exist_ok=True)
#
# print('Upload : insightflow_synthetic_full.csv')
# uploaded = files.upload()
# for fname in uploaded:
#     os.rename(fname, f'InsightFlowExecutive/ml/dataset/{fname}')
#
# print('Upload : finetune.py')
# uploaded2 = files.upload()
# os.rename('finetune.py', 'InsightFlowExecutive/ml/finetune.py')
# %cd InsightFlowExecutive

In [ ]:
# ── Étape 4 : Vérifier le dataset ─────────────────────────────
import csv
from collections import Counter

path = 'ml/dataset/insightflow_synthetic_full.csv'
rows = list(csv.DictReader(open(path, encoding='utf-8')))

sentiment_counts = Counter(r['sentiment_label'] for r in rows if r.get('sentiment_label') in ('POSITIVE','NEUTRAL','NEGATIVE'))
emotion_counts   = Counter(r['emotion_label']   for r in rows if r.get('emotion_label')   in ('frustration','concern','urgency','neutral','satisfaction'))

print(f'Total rows        : {len(rows)}')
print(f'Sentiment samples : {sum(sentiment_counts.values())} → {dict(sentiment_counts)}')
print(f'Emotion samples   : {sum(emotion_counts.values())}   → {dict(emotion_counts)}')

---
## Fine-tuning Sentiment — XLM-RoBERTa (FR+EN)
**Durée estimée : ~25 min sur T4**

In [ ]:
# ── Fine-tune XLM-RoBERTa pour le sentiment ───────────────────
!python ml/finetune.py --task sentiment --model xlm --lang full

---
## Fine-tuning Émotion — distilRoBERTa (5 classes business)
**Durée estimée : ~15 min sur T4**

In [ ]:
# ── Fine-tune distilRoBERTa pour l'émotion ────────────────────
!python ml/finetune.py --task emotion --lang full

---
## Résultats comparatifs

In [ ]:
# ── Afficher les résultats des deux modèles ───────────────────
import json, os

print('=' * 65)
print('  InsightFlow — Fine-tuning Results')
print('=' * 65)

models = [
    ('insightflow-sentiment-xlm-v1', 'Sentiment — XLM-RoBERTa (FR+EN)'),
    ('insightflow-emotion-v1',       'Émotion   — distilRoBERTa (5 classes)'),
]

for model_dir, label in models:
    path = f'ml/models/{model_dir}/training_summary.json'
    if os.path.exists(path):
        s = json.load(open(path))
        print(f'\n  {label}')
        print(f'  Base     : {s["model_base"].split("/")[-1]}')
        print(f'  Dataset  : {s["dataset_size"]} samples | Lang : {s["language"]}')
        print(f'  Accuracy : {s["eval_accuracy"]*100:.2f}%')
        print(f'  F1-macro : {s["eval_f1_macro"]*100:.2f}%')
    else:
        print(f'\n  {label} — pas encore entraîné')

print('\n' + '=' * 65)

---
## Télécharger les modèles fine-tunés

In [ ]:
# ── Télécharger les modèles fine-tunés ────────────────────────
import shutil
from google.colab import files

for model_dir in ['insightflow-sentiment-xlm-v1', 'insightflow-emotion-v1']:
    path = f'ml/models/{model_dir}'
    if os.path.exists(path):
        zip_path = f'{model_dir}.zip'
        shutil.make_archive(model_dir, 'zip', 'ml/models', model_dir)
        files.download(zip_path)
        print(f'✓ Téléchargé : {zip_path}')
    else:
        print(f'✗ Modèle non trouvé : {model_dir}')